# CSA trên Q-Former — pilot hai pha, T4×2
**Internet ON, GPU T4×2, Save & Run All. Một notebook dùng cả hai GPU.**
Pha1:1250updates chỉ học core CSA bằng CE, LRwarmup100steps lên1e-4, baseline frozen.
Pha2:1250updates học chung, LRbase5e-6/core2e-5, CE200steps rồi CE+.05SCST;
rewardCIDErD+.5object_count_v2. Reset optimizer/RNG/sampler giữa hai pha, bank luôn frozen.
Giữ core2object+1relation và memory[baseline;Vo;Vr] đã smoke. Không dùng checkpoint sau smoke.
Train10k/dev1000 như A, seed42, globalbatch8. Chỉ cuối pha2 đánh giá original/on/off rawgreedy.
Không test/rerank/best-checkpoint selection; CSA có thêm1250core-onlysteps soA.
Dataset giống CSA smoke; không cần cache UPT hoặc VG. Hai bank/sourceCSA tải có hash kiểm.

In [ ]:
print('CSA TWO-PHASE PILOT: BOOTSTRAP START',flush=True)
from pathlib import Path
CHECKPOINT=Path('/kaggle/input/datasets/ducanh2403/qformer-scst-object-w05-legacy/qformer_scst_object_w05_legacy.pth')
LEGACY_PROMPT=Path('/kaggle/input/datasets/ducanh2403/prompt-cache/prompt_clip_tokens_cache.pt')
VISUAL_CACHE=Path('/kaggle/input/datasets/ducanh2403/visual-cache')
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
DETECTIONS=Path('/kaggle/input/datasets/ducanh2403/objectdetectionecache/objectdetectioncache.json')
OUTPUT=Path('/kaggle/working/csa_two_phase_pilot')
PHASE1_CHECKPOINT=None  # Chỉ khi tiếp tục từ phase1_checkpoint.bin của đúng notebook này.
for p in (CHECKPOINT,LEGACY_PROMPT,COCO_JSON,DETECTIONS):
    assert p.is_file(),p
assert VISUAL_CACHE.is_dir(),VISUAL_CACHE
assert not OUTPUT.exists(),'Preserve existing output; use fresh OUTPUT/session.'
if PHASE1_CHECKPOINT is not None:
    PHASE1_CHECKPOINT=Path(PHASE1_CHECKPOINT)
    assert PHASE1_CHECKPOINT.is_file(),PHASE1_CHECKPOINT
OUTPUT.mkdir(parents=True)


In [ ]:
import base64, hashlib, json, os, shutil, subprocess, sys, tempfile, zlib
os.environ.update(PYTHONUNBUFFERED='1', HDF5_USE_FILE_LOCKING='FALSE', PYTHONHASHSEED='42',
    CUBLAS_WORKSPACE_CONFIG=':4096:8', OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2', TOKENIZERS_PARALLELISM='false', HF_HUB_DISABLE_XET='1')
# Isolate each bootstrap: embedded tracked overlays from a prior run are intentional edits.
REPO = Path(tempfile.mkdtemp(prefix='Image_Captioning-hidden-csa_pilot-'))/'repo'
LOG = Path('/tmp/upt_hidden_csa_pilot.log')
PROJECT_COMMIT = 'dfa1d4e0c52bea8672c6b770127034c2ec377b58'
def run(args, cwd=None):
    args = list(map(str, args))
    print('\nRunning:', ' '.join(args), flush=True)
    with LOG.open('a', encoding='utf-8') as log:
        log.write('\nRunning: ' + ' '.join(args) + '\n'); log.flush()
        with subprocess.Popen(args, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                text=True, errors='replace', bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True); log.write(line); log.flush()
            status = process.wait()
    if status:
        raise RuntimeError(f'Command failed ({status}). Saved log: {LOG}')
def checkout(url, path, revision):
    if not path.exists():
        run(['git','clone',url,path])
    assert (path/'.git').is_dir(), path
    assert not subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=path,text=True).strip(), 'Preserve tracked edits; use a fresh runtime path.'
    run(['git','fetch','--depth','1','origin',revision], cwd=path)
    run(['git','checkout','--detach',revision], cwd=path)
checkout('https://github.com/Supzxjee/Image_Captioning.git', REPO, PROJECT_COMMIT)
SOURCES = json.loads(zlib.decompress(base64.b64decode('...')).decode('utf-8'))
SOURCE_HASHES = {'pilot_data.py': '93fa39b1de581d774417840be4d5e0b2b8d52bbf9ee53b479adad2c60a45d3d7', 'pilot_model.py': '879046e943169c30e640fd9b04d6a1edfccd19cd17b5311b104b9ec2dc5bf8f1', 'run_pilot.py': 'c34b7d7f2aae7e050190f713370d0c5f85b283bf342e328e63baeaf2783061a1', 'hidden_data.py': '9285d02c5aa7660a5656654e2efe034941bc24d6f7c300958c06fcca6f0d43c7', 'captioning/scst_runtime.py': '3452ccfbd43876dc6381b78d1229d1d6b05fe723e3863a04f31c4a784497f669', 'hidden_metric_smoke.py': '8cf64c56b169cdd2407fdeb668408a80d947578632154405c47c7040558a698b', 'csa_core.py': '45ee64e8b1f78e772e3b788f647895e8377bd4a28ce4e3cfd871e12f756759a1', 'csa_model.py': '68e5a5dad4a8ba2df31cdb36a6d8d8c65c28b0cccb9bbd17c656512482b0da44', 'phase_policy.py': '5aca7edad92a85f6acdc692591777a75b10870a98330f7d396efa830dc3a4efc', 'run_csa_pilot.py': '916e107ee56e134fe6e320eb4a7c5041e8f4c2dfb620460bffde34a832b75f66', 'test_policy_static.py': 'f11f7e81e70cd43bb9117b42ff9de61cd9fc95c88f0d45467b4b1349166886a5', 'test_phase_model.py': '1911d247f936c8ecf14fd9f899b6396de4b03dcefa783728f301d880e2fb45d9'}
for name, source in SOURCES.items():
    assert hashlib.sha256(source.encode()).hexdigest() == SOURCE_HASHES[name], name
    destination = REPO/name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(source, encoding='utf-8')
print('Pinned runtime and hash-checked embedded source READY', flush=True)


In [ ]:
run([sys.executable,'-m','pip','install','-q','transformers==4.57.1','pycocotools','pycocoevalcap==1.2',
     'h5py','numpy','Pillow','tqdm','nltk==3.9.2','future'])
run([sys.executable,'-m','pip','install','-q','--no-deps',
     'git+https://github.com/clips/pattern.git@af754685cca3713db0abc4f020f2e94467c19d85'])
run([sys.executable,'-u','-c',"import nltk;from pattern.en import singularize;assert singularize('umbrellas')=='umbrella';assert nltk.download('punkt');assert nltk.download('punkt_tab')"])
run(['java','-version'])
import torch
assert torch.cuda.device_count()==2,'Select T4x2.'
CHAIR=Path('/tmp/csa-pilot-chair')
checkout('https://github.com/Maxlinn/CHAIR-metric-standalone.git',CHAIR,'4087a26211aa2339b9a76307cb8f0321ef691d0a')


In [ ]:
import urllib.request,zipfile
sys.path.insert(0,str(REPO))
from csa_core import BANKS,REFERENCE_SHA,sha
REFERENCE=OUTPUT/'reference'
REFERENCE.mkdir()
REF_COMMIT='0cc570ba7c50ef08c46a21439fe73ef0a5df97ab'
assets={'models/caption/csa_net.py':REFERENCE_SHA}
assets.update({'coco_caption/'+name:value[0] for name,value in BANKS.items()})
for name,source in SOURCES.items():
    p=OUTPUT/'source'/name
    p.parent.mkdir(parents=True,exist_ok=True)
    p.write_text(source,encoding='utf-8')
CONFIG=OUTPUT/'requested_config.json'
cfg=dict(arm='CSA',checkpoint=str(CHECKPOINT),legacy_prompt=str(LEGACY_PROMPT),
         visual_cache=str(VISUAL_CACHE),dataset_json=str(COCO_JSON),detections=str(DETECTIONS),
         chair=str(CHAIR),reference=str(REFERENCE),output=str(OUTPUT),stage='/tmp/csa_two_phase_stage.h5',
         phase1_checkpoint=str(PHASE1_CHECKPOINT or OUTPUT/'phase1_checkpoint.bin'),
         resume_phase1=PHASE1_CHECKPOINT is not None,project_commit=PROJECT_COMMIT,csa_commit=REF_COMMIT,
         implementation_sha256=SOURCE_HASHES,global_batch_size=8,updates_per_phase=1250,
         evaluation_policy='Fixed final phase2 checkpoint; original/on/off greedy; reused dev1000, no test.',
         gate=dict(CIDEr_delta_vs_A_min=.01,paired95_lower_gt=0,CHAIRs_increase_max=0,
                   CHAIRi_increase_max=0,Recall_drop_max=.01,beat_original=True),
         comparison_limits='1250 extra core-only updates vs A; not matched compute/capacity.')
CONFIG.write_text(json.dumps(cfg,indent=2),encoding='utf-8')
PROTOCOL=OUTPUT/'protocol.json'
try:
    for name,expected in assets.items():
        print('Downloading pinned author asset:',name,flush=True)
        url=f'https://raw.githubusercontent.com/CrossmodalGroup/CSA-Net/{REF_COMMIT}/{name}'
        request=urllib.request.Request(url,headers={'User-Agent':'csa-two-phase-pilot'})
        with urllib.request.urlopen(request,timeout=120) as response:
            (REFERENCE/Path(name).name).write_bytes(response.read())
        assert sha(REFERENCE/Path(name).name)==expected,name
    run([sys.executable,'-u','test_policy_static.py'],cwd=REPO)
    run([sys.executable,'-u','test_phase_model.py'],cwd=REPO)
    for phase in (1,2):
        run([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1',
             '--nproc_per_node=2','--max_restarts=0','test_phase_model.py','--phase',phase],cwd=REPO)
    run([sys.executable,'-u','hidden_metric_smoke.py',CHAIR],cwd=REPO)
    run([sys.executable,'-u','run_csa_pilot.py','prepare','--config',CONFIG],cwd=REPO)
    phases=(2,) if PHASE1_CHECKPOINT is not None else (1,2)
    for phase in phases:
        run([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1',
             '--nproc_per_node=2','--max_restarts=0','run_csa_pilot.py','worker',
             '--phase',phase,'--config',PROTOCOL],cwd=REPO)
    run([sys.executable,'-u','run_csa_pilot.py','evaluate','--config',PROTOCOL],cwd=REPO)
finally:
    if LOG.is_file():
        shutil.copyfile(LOG,OUTPUT/'notebook.log')
    if PROTOCOL.is_file():
        run([sys.executable,'-u','run_csa_pilot.py','archive','--config',PROTOCOL],cwd=REPO)
    else:
        with zipfile.ZipFile(str(OUTPUT)+'_review.zip','w',zipfile.ZIP_DEFLATED) as z:
            for p in sorted(OUTPUT.rglob('*')):
                if p.is_file() and p.suffix in ('.json','.py','.log'):
                    z.write(p,p.relative_to(OUTPUT))
print('SEND REVIEW:',str(OUTPUT)+'_review.zip',flush=True)
print('KEEP FINAL CHECKPOINT:',OUTPUT/'checkpoint.bin',flush=True)
print('KEEP PHASE1 CHECKPOINT:',cfg['phase1_checkpoint'],flush=True)


Gửi `csa_two_phase_pilot_review.zip`; giữ riêng `checkpoint.bin` và `phase1_checkpoint.bin`.
Nếu hết phiên sau pha1, mount checkpoint đó vào notebook mới, sửa PHASE1_CHECKPOINT ở cellđầu.
Pha2 vẫn tạo optimizer mới và reset RNG/sampler; dữ liệu/source/recipe khác sẽ bị từ chối.
Không dùng checkpoint.bin cuối pha2 hoặc checkpoint smoke làm checkpoint pha1.
Đánh giá/soA sau khi kiểm matched protocol/runtime; chưa tự động tuyên bố gain/grounding.
Chỉ khi ΔCIDEr≥.01 và paired95%lower>0, CHAIRs/i không tăng, recalldrop≤.01, vượtoriginal
mới review mù factuality; không đạt thì dừng recipe, không tự fulltrain/tune/extendsteps.
Nếu lỗi: `print(LOG.read_text(encoding='utf-8')[-16000:])`.